# Обучение ML-модели

##Чтение данных:

In [1]:
!pip install boto3

In [2]:
import boto3
import pandas as pd
import os
from google.colab import userdata

In [3]:
os.environ["KEY_ID"] = userdata.get("KEY_ID")
os.environ["ACCESS_KEY"] = userdata.get("ACCESS_KEY")

KEY_ID = os.environ["KEY_ID"]
ACCESS_KEY = os.environ["ACCESS_KEY"]

In [4]:
S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/listings.csv"

In [5]:
s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    aws_access_key_id=KEY_ID,
    aws_secret_access_key=ACCESS_KEY,
    region_name="ru-central1",
)

In [6]:
def read_s3_csv(bucket, key):
    response = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(response["Body"])

In [7]:
df_s3 = read_s3_csv(S3_BUCKET, S3_KEY)
display(df_s3.head())
print("Размер таблицы:", df_s3.shape)
print("Колонки:", df_s3.columns.tolist())

,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


Размер таблицы: (20151, 13)
Колонки: ['listing_id', 'snapshot_date', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'neighbourhood_cleansed', 'property_type', 'room_type', 'price']


##Исследование датасета:

In [8]:
df_s3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20151 entries, 0 to 20150
Data columns (total 13 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   listing_id              20151 non-null  int64  
 1   snapshot_date           20151 non-null  object 
 2   latitude                20151 non-null  float64
 3   longitude               20151 non-null  float64
 4   accommodates            20151 non-null  int64  
 5   bathrooms               20120 non-null  float64
 6   bedrooms                18603 non-null  float64
 7   beds                    19469 non-null  float64
 8   minimum_nights          20111 non-null  float64
 9   neighbourhood_cleansed  20151 non-null  object 
 10  property_type           20151 non-null  object 
 11  room_type               20151 non-null  object 
 12  price                   20151 non-null  float64
dtypes: float64(7), int64(2), object(4)
memory usage: 2.0+ MB


In [9]:
df_s3.isna().sum()

,0
listing_id,0
snapshot_date,0
latitude,0
longitude,0
accommodates,0
bathrooms,31
bedrooms,1548
beds,682
minimum_nights,40
neighbourhood_cleansed,0


In [10]:
df_s3.describe()

,listing_id,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,price
count,2.015100e+04,20151.000000,20151.000000,20151.000000,20120.000000,18603.000000,19469.000000,20111.000000,20151.000000
mean,1.023943e+18,51.507860,-0.134513,3.557193,1.354970,1.609740,2.007910,3.185073,250.966359
std,5.986249e+17,0.050673,0.104143,2.247511,0.751998,1.014536,1.427495,9.276967,484.703977
min,1.540000e+04,51.306650,-0.496760,1.000000,0.000000,0.000000,0.000000,1.000000,4.320000
25%,6.908038e+17,51.485040,-0.193483,2.000000,1.000000,1.000000,1.000000,1.000000,99.550000
50%,1.205084e+18,51.512862,-0.136727,3.000000,1.000000,1.000000,2.000000,2.000000,174.000000
75%,1.520169e+18,51.535691,-0.075190,4.000000,1.500000,2.000000,3.000000,3.000000,286.000000
max,1.776091e+18,51.678772,0.236390,16.000000,12.500000,15.000000,24.000000,365.000000,23686.000000


In [11]:
df_s3.duplicated().sum()

np.int64(960)

Датасет содержит информацию об объектах аренды. В рамках задания будет решаться задача **регрессии** (предсказание стоимости аренды `price` (целевая переменная) на основе характеристик объекта).

Датасет содержит 20151 стоку и 13 столбцов. Данные содержат числовые и категориальные признаки. Обнаружены пропущенные значения в столбцах `bathrooms`, `bedrooms`, `beds` и `minimum_nights` и 960 полных дубликатов. В целевой переменной пропуски отсутствуют. Распределение целевой переменной говорит о наличии экстремально высоких значений.

Перед разделением данных будут удалены полные дубликаты.
Обработка пропусков и кодирование категориальных
признаков после разделения на train и test, чтобы избежать утечки данных.

In [12]:
df = df_s3.drop_duplicates()

##Разделение данных:

Целевая переменная: `price`. Идентификатор объявления (`listing_id`) и дата сбора данных (`snapshot_date`) не используются в качестве признаков.

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
X = df[["latitude", "longitude", "accommodates", "bathrooms", "bedrooms", "beds", "minimum_nights", "neighbourhood_cleansed", "property_type", "room_type"]]
y = df["price"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##Преобразование признаков:

Пропуски в числовых признаках будут заполнены медианой с помощью `SimpleImputer`. Категориальные признаки закодированы с помощью `OneHotEncoder`.

Преобразования объединяются с помощью `ColumnTransformer`
и обучаются только на обучающей выборке во избежание утечки данных. К тестовой выборке применяются уже обученные преобразования.

In [17]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

In [18]:
numeric_columns = ["latitude", "longitude", "accommodates", "bathrooms", "bedrooms", "beds", "minimum_nights"]
category_columns = ["neighbourhood_cleansed", "property_type", "room_type"]

In [19]:
numeric_pipeline = Pipeline([('fill', SimpleImputer(strategy='median'))])

In [20]:
categorical_pipeline = Pipeline([('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

In [21]:
preprocessor = ColumnTransformer([('numbers', numeric_pipeline, numeric_columns), ('category', categorical_pipeline, category_columns)], remainder='drop')

In [22]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

##Обучение ML-модели:

Для решения задачи регрессии выбрана модель **RandomForestRegressor**, из-за простоты реализации, способности учитывать нелинейные зависимости между признаками и отсутствие необходимости масштабирования числовых признаков.

In [23]:
from sklearn.ensemble import RandomForestRegressor

In [24]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_processed, y_train)

RandomForestRegressor(random_state=42)

##Оценка качества модели:

Качество модели будет оцениваться на тестовой выборке с помощью стандартных метрик `MAE`, `RMSE` и `R2`.

In [25]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

In [26]:
y_pred = model.predict(X_test_processed)

In [27]:
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

In [28]:
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 3))

MAE: 66.64
RMSE: 246.73
R2: 0.494


Модель объясняет около половины вариации цен (умеренное качество предсказания). В среднем предсказанная стоимость отличается от фактической на 66,64 денежной единицы. Превышение RMSE над MAE означает, что для некоторых объектов модель допускает существенные ошибки. Это может быть связано с наличием экстремально высоких цен в исходных данных.